In [83]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
import sys
import os
import numpy as np
from datetime import datetime
np.NaN=np.nan
import pandas_ta as ta
import importlib
from typing import Tuple
import vectorbt as vbt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tqdm 
from tabulate import tabulate
import warnings
import re
import time
from tvDatafeed import TvDatafeed, Interval

warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)


In [85]:
import pandas as pd

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000) 
pd.set_option('display.max_colwidth', None)

In [86]:
#Input file
INPUT_FILE='split_ohlcv_data/all_ohlcv.csv'
#General variables
START_DATE = "2017-01-01"
TRADING_START_DATE = "2018-01-01"
END_DATE = "2026-01-01"

# Zscore generation
ROLLING_THRESHOLD     = -99999
ROLLING_PERIOD        = 16
MEAN_PERIOD           = 28

# Weights Generation
Z_MEAN                = True
FREQUENCY             = 'monthly'
INITIAL_CAPITAL       = 100000
NUM_STOCKS_ACTIVE     = 30
ZSCORE_THRESHOLD      = 0.1
ZSCORE_PERIOD         = 24

# Zscore & Weights----------------------- 


In [87]:
# Preprocessing: Generate required files from all_ohlcv.csv

import os

# Create directories if they don't exist
os.makedirs('split_ohlcv_data', exist_ok=True)
os.makedirs('src', exist_ok=True)

# Load the master OHLCV data
print("Loading all_ohlcv.csv...")
master_df = pd.read_csv(INPUT_FILE, index_col=0, parse_dates=True, header=[0, 1])

# Save to split_ohlcv_data
master_df.to_csv('split_ohlcv_data/all_ohlcv.csv')
print("   -> Created split_ohlcv_data/all_ohlcv.csv")

# Save the clipped version (same as master since no clipping)
master_df.to_csv('src/all_ohlcv_clipped.csv')
print("   -> Created src/all_ohlcv_clipped.csv")

# Split into individual attribute files
print("Splitting into individual attribute files...")
attributes = ['open', 'high', 'low', 'close', 'volume']
for attr in attributes:
    attr_df = master_df.xs(attr, axis=1, level=1)
    attr_df.index.name = 'Date'
    output_file = f"split_ohlcv_data/all_{attr}.csv"
    attr_df.to_csv(output_file)
    print(f"   -> Created {output_file}")

# Create all_close_dateonly.csv (same as all_close.csv but ensure date index)
close_df = master_df.xs('close', axis=1, level=1)
close_df.index.name = 'Date'
close_df.to_csv('src/all_close_dateonly.csv')
print("   -> Created src/all_close_dateonly.csv")

# Create cleaned_weekly_data.csv (weekly aggregated closes)
print("Creating weekly data...")
weekly_close = close_df.resample('W').last()  # Take last price of each week
weekly_close.index.name = 'Date'
weekly_close.to_csv('src/cleaned_weekly_data.csv')
print("   -> Created src/cleaned_weekly_data.csv")

# Generate date_mapping.csv
print("Generating date mapping...")
# Create a date range from the data
start_date = close_df.index.min()
end_date = close_df.index.max()

# Generate weekly periods
weekly_periods = pd.date_range(start=start_date, end=end_date, freq='W')

date_mapping_rows = []
for i, week_end in enumerate(weekly_periods):
    week_start = week_end - pd.Timedelta(days=6)  # Assuming Monday to Sunday week
    rebalancing_date = week_end  # Use week_end as rebalancing date
    date_mapping_rows.append({
        'rebalancing_date': rebalancing_date,
        'week_start': week_start,
        'week_end': week_end
    })

date_mapping_df = pd.DataFrame(date_mapping_rows)
date_mapping_df.to_csv('src/date_mapping.csv', index=False)
print("   -> Created src/date_mapping.csv")

print("Preprocessing completed!")

Loading all_ohlcv.csv...
   -> Created split_ohlcv_data/all_ohlcv.csv
   -> Created src/all_ohlcv_clipped.csv
Splitting into individual attribute files...
   -> Created split_ohlcv_data/all_open.csv
   -> Created split_ohlcv_data/all_high.csv
   -> Created split_ohlcv_data/all_low.csv
   -> Created split_ohlcv_data/all_close.csv
   -> Created split_ohlcv_data/all_volume.csv
   -> Created src/all_close_dateonly.csv
Creating weekly data...
   -> Created src/cleaned_weekly_data.csv
Generating date mapping...
   -> Created src/date_mapping.csv
Preprocessing completed!


In [88]:
def z_score(rolling_threshold, period, mean_period, mode='normal'):
    # Load weekly close data and extract symbols
    weekly_close = pd.read_csv('src/cleaned_weekly_data.csv', parse_dates=['Date'], index_col='Date')
    
    #zscore calculation
    weekly_returns = np.log(weekly_close / weekly_close.shift(1))
    rolling_mean = weekly_returns.rolling(window=period).mean()
    rolling_std = weekly_returns.rolling(window=period).std()

    # Calculate z-scores with proper handling of division by zero
    # Add small epsilon to prevent division by zero
    epsilon = 1e-10
    rolling_std_safe = rolling_std.replace(0, epsilon)
    
    # Calculate z-scores safely
    z_scores = (weekly_returns - rolling_mean) / rolling_std_safe
    
    # Create the Z-Scores DataFrame
    z_scores_df = pd.DataFrame(z_scores, index=weekly_returns.index, columns=weekly_returns.columns)
    
    # Replace inf and -inf with NaN, then fill NaN with 0
    z_scores_df = z_scores_df.replace([np.inf, -np.inf], np.nan)
    z_scores_df = z_scores_df.fillna(0)
    
    # Compute rolling mean of period 'mean_period' for Z-scores
    z_scores_mean_df = z_scores_df.rolling(window=mean_period).mean()
    
    # Handle inf values only (keep NaN as NaN for warm-up period)
    z_scores_mean_df = z_scores_mean_df.replace([np.inf, -np.inf], np.nan)
    
    # Do NOT fill NaN values in the rolling mean during warm-up period
    # Each mean_period will naturally have NaN for its first (mean_period-1) rows
    # This ensures distinct behavior: mean_period=20 starts giving signals at row 20,
    # mean_period=24 starts at row 24, etc.
    # The stock selection logic will handle NaN by using dropna()
    
    if mode=='normal':
        z_scores_df.to_csv('auxilary/z_scores.csv')
        z_scores_mean_df.to_csv('auxilary/z_scores_mean.csv')
        print("Z-Score calculation completed and files saved.")
    return z_scores_df, z_scores_mean_df


In [89]:
def record_monthly_weights(weights_df, current_date, next_date, top_set, total_active_sets):
    """
    On the rebalance date: find the nearest valid trading day and assign weight = 1/N
    On the next rebalance date: assign weight = 0 (so dropped stocks are closed out)
    """
    if not top_set:
        return

    # 1) Find the actual start trading day (first available date on or after current_date)
    valid_starts = weights_df.index[weights_df.index >= current_date]
    if valid_starts.empty:
        return
    start_trading_date = valid_starts[0]
    
    # 2) Assign 1/N on this valid start date
    if total_active_sets: 
        w = 1.0 / total_active_sets
        weights_df.loc[start_trading_date, list(top_set)] = w
    else:
        weights_df.loc[start_trading_date, list(top_set)] = np.nan

    # 3) Find the actual next trading day (first available date on or after next_date)
    valid_nexts = weights_df.index[weights_df.index >= next_date]
    if not valid_nexts.empty:
        next_trading_date = valid_nexts[0]
        # 4) Assign 0.0 on the next rebalance date 
        # (If the stock is selected again, it will be overwritten with the new weight in the next iteration)
        weights_df.loc[next_trading_date, list(top_set)] = 0.0
def get_rebalancing_dates(mapping_csv="src/date_mapping.csv"):
    # Load mapping
    df = pd.read_csv(mapping_csv, parse_dates=["rebalancing_date", "week_start", "week_end"])
    # Ensure sorted by Week_End
    df = df.sort_values("week_end").drop_duplicates("week_end")
    # Extract first week_end of each month
    monthly_rebalancing = df.groupby([df["week_end"].dt.year, df["week_end"].dt.month])["week_end"].first().sort_values().to_list()
    return monthly_rebalancing
def weights_generation(
    start_date, 
    end_date, 
    trading_start_date,
    z_mean,
    z_scores_df,
    z_scores_mean_df,
    # --- Optional Arguments ---
    initial_capital=100000, 
    number_stocks_active=100, 
    zscore_threshold=0.0,
    mode='incremental'
):
    # Load stock prices and extract symbols
    stock_prices = pd.read_csv('src/all_close_dateonly.csv', parse_dates=['Date'], index_col='Date')
    
    dates = stock_prices
    daily_index = dates.index
    daily_index = daily_index[1:]
    daily_index = pd.DatetimeIndex(daily_index)

    weights_df = pd.DataFrame(
        data    = np.nan,
        index   = daily_index,
        columns = dates.columns  # all tickers in that file
    )
    
    # Extract universe (symbols) directly from the data
    universe = dates.columns.tolist()
    
    if z_mean: z_scores=z_scores_mean_df.copy()
    else: z_scores=z_scores_df.copy()
    # --- FIX 1: CLEAN THE Z_SCORES INDEX ---
    # This is critical for `looptest` mode where the cache file may be messy
    z_scores = z_scores[z_scores.index.notna()]
    try:
        z_scores.index = z_scores.index.tz_localize(None)
    except TypeError:
        pass # Already naive
    z_scores.sort_index(inplace=True)
    z_scores = z_scores[~z_scores.index.duplicated(keep='first')]
    # --- END OF CLEANING ---
    # Filter by main date range
    stock_prices = stock_prices.loc[start_date:end_date]
    # We must NOT slice by start_date here. We need the historical data for lookbacks (z_date). We only slice by end_date.
    z_scores = z_scores.loc[:end_date]

    if z_scores.empty:
        print("Warning: Z-scores DataFrame is empty after loading/slicing by universe.")
        raise ValueError("No z-score data available for the specified period or universe.")
    # Load date_mapping *before* reindexing to find the true earliest date
    date_mapping = pd.read_csv("src/date_mapping.csv", parse_dates=["week_start", "week_end", "rebalancing_date"])
    
    # Find the earliest 'week_start' date from mapping. This is the
    # absolute earliest date we might ever need to look up for a z_date.
    earliest_z_date_needed = date_mapping['week_start'].min()
    
    # Create a full daily index from this *true* earliest date to the end_date.
    # This index will now correctly include the lookback period (e.g., '2023-12-26')
    full_date_index = pd.date_range(start=earliest_z_date_needed, end=end_date, freq='D')

    # Reindex stock_prices. It will have NaNs before start_date, which is fine.
    stock_prices = stock_prices.reindex(full_date_index, method='ffill')
    
    # Reindex z_scores. This will fill all gaps.
    # Dates *before* the first date in the z_score file will be NaN, 
    # as ffill() cannot fill backwards. This is correct.
    z_scores = z_scores.reindex(full_date_index, method='ffill')
    
    if mode == 'incremental':
        print('Stock prices and Z-scores loaded and reindexed.')

    portfolio_value = initial_capital
    portfolio_history = []
    
    # --- 3. Set Up Rebalancing Dates ---
    
    # Relies on global `get_rebalancing_dates` function
    rebalancing_dates = get_rebalancing_dates(mapping_csv='src/date_mapping.csv') 
    

    rebalancing_dates = pd.DatetimeIndex(rebalancing_dates)
    print(f"Rebalancing dates: {rebalancing_dates[-1]}")

    # Filter dates based on the provided parameters
    rebalancing_dates = rebalancing_dates[rebalancing_dates >= trading_start_date]
    rebalancing_dates = rebalancing_dates[(rebalancing_dates >= start_date) & (rebalancing_dates <= end_date)]
    rebalancing_dates = pd.DatetimeIndex(rebalancing_dates)
    
    if len(rebalancing_dates) < 2:
        print("Not enough rebalancing dates in the specified period to run backtest.")
        return

    portfolio_history.append({'date': rebalancing_dates[0], 'value': portfolio_value})
    prev_top_stocks = set()

    for i, current_date in enumerate(rebalancing_dates[:-1]):
        next_date = rebalancing_dates[i + 1]

        # --- ROBUST DATE MAPPING ---
        # Find the latest mapping row on or before the current_date
        mapping_subset = date_mapping[date_mapping["rebalancing_date"] <= current_date]
        if mapping_subset.empty:
            print(f"Warning: No date mapping found on or before date {current_date}. Skipping.")
            continue
            
        # Get the latest valid mapping entry. We assume the last one is the correct one. This handles cases where current_date is not in mapping, using the one just before it.
        matched_mapping_row = mapping_subset.iloc[-1]
        current_week_start = matched_mapping_row["week_start"]
        current_week_end = matched_mapping_row["week_end"]

        # For previous_distinct_week_start, we find the latest one *before* the current_week_start
        prev_mapping_subset = date_mapping[date_mapping["week_start"] < current_week_start]
        if prev_mapping_subset.empty:
            print(f"Warning: No prior week_start found for date {current_date}. Skipping.")
            continue
        previous_distinct_week_start = prev_mapping_subset.iloc[-1]["week_start"]
        # --- END ROBUST DATE MAPPING ---

        if mode == 'incremental':
            print(f"{current_date} --> {previous_distinct_week_start}")
            
        if(current_date==current_week_end) :
            z_date= current_week_start
        
        else :
            z_date= previous_distinct_week_start
        
        current_z_scores = z_scores.loc[z_date]
        
        # Remove inf, -inf, and NaN values
        current_z_scores = current_z_scores.replace([np.inf, -np.inf], np.nan).dropna()
        
        if mode == 'incremental':
            print(f"Regular Rebalance: {current_date} --> Using Z-Scores from {z_date}")
            print(f"Valid z-scores available: {len(current_z_scores)}")
            
        # Select stocks above the threshold
        qualified_stocks = current_z_scores[current_z_scores >= zscore_threshold]
        
        if len(qualified_stocks) == 0:
            if mode == 'incremental':
                print(f"Warning: No stocks meet threshold on {current_date}. Skipping rebalance.")
            continue
        
        top_stocks = qualified_stocks.nlargest(min(number_stocks_active, len(qualified_stocks))).index
        
        top_set = set(top_stocks)
        if mode == 'incremental':
            retained_stocks = top_set & prev_top_stocks
            expelled_stocks = prev_top_stocks - top_set
            new_additions = top_set - prev_top_stocks

            print(f"\nRebalancing on {current_date}:")
            print(f"Qualified stocks: {list(top_stocks)}")
            print(f"Retained stocks: {list(retained_stocks)}")
            print(f"Expelled stocks: {list(expelled_stocks)}")
            print(f"New additions: {list(new_additions)}")

        prev_top_stocks = top_set
        
        # Relies on global `record_monthly_weights` function
        record_monthly_weights(weights_df, current_date,next_date, top_set,number_stocks_active)
    if mode == 'looptest':
        weights_df.to_csv(".cache/backtester_weights_cache.csv", index_label="Date")
    elif mode == 'incremental':
        # --- ROBUST FINAL DATE MAPPING ---
        final_date = rebalancing_dates[-1]
        
        final_mapping_subset = date_mapping[date_mapping["rebalancing_date"] <= final_date]
        if final_mapping_subset.empty:
             print(f"Warning: No date mapping found for final date {final_date}. Cannot save final weights.")
             return # Or handle error differently
        
        final_matched_row = final_mapping_subset.iloc[-1]
        final_week_start = final_matched_row["week_start"]
        # --- END ROBUST FINAL DATE MAPPING ---
        
        # Safe call thanks to ffill
        final_z_scores = z_scores.loc[final_week_start].dropna()
        final_top_stocks = final_z_scores[final_z_scores >= zscore_threshold].nlargest(number_stocks_active).index

        final_top_set = set(final_top_stocks)
        final_retained = final_top_set & prev_top_stocks

        final_expelled = prev_top_stocks - final_top_set
        final_expelled = prev_top_stocks - final_top_set

        final_new = final_top_set - prev_top_stocks
        final_new = final_top_set - prev_top_stocks




        print(f"\nFinal Rebalancing on {final_date}:")
        print(f"\nFinal Rebalancing on {final_date}:")

        print(f"Qualified stocks: {list(final_top_stocks)}")
        print(f"Qualified stocks: {list(final_top_stocks)}")        
        weights_df.to_csv("auxilary/backtester_weights.csv", index_label="Date")

        print(f"Retained stocks: {list(final_retained)}")
        print(f"Retained stocks: {list(final_retained)}")        
        weights_df.to_csv("auxilary/backtester_weights.csv", index_label="Date")

        print(f"Expelled stocks: {list(final_expelled)}")
        print(f"Expelled stocks: {list(final_expelled)}")        
        print(f"New additions: {list(final_new)}")
        print(f"New additions: {list(final_new)}")

In [90]:
z_scores_df, z_scores_mean_df = z_score(ROLLING_THRESHOLD, ROLLING_PERIOD, MEAN_PERIOD)
weights_generation(START_DATE, END_DATE,TRADING_START_DATE, Z_MEAN,z_scores_df,z_scores_mean_df,INITIAL_CAPITAL, NUM_STOCKS_ACTIVE,ZSCORE_THRESHOLD,mode='incremental')

Z-Score calculation completed and files saved.
Stock prices and Z-scores loaded and reindexed.
Rebalancing dates: 2025-12-07 09:15:00
2018-01-07 09:15:00 --> 2017-12-25 09:15:00
Regular Rebalance: 2018-01-07 09:15:00 --> Using Z-Scores from 2018-01-01 09:15:00
Valid z-scores available: 496

Rebalancing on 2018-01-07 09:15:00:
Qualified stocks: ['GPIL.NS', 'LTTS.NS', 'SONATSOFTW.NS', 'ABBOTINDIA.NS', 'JWL.NS', 'JUSTDIAL.NS', 'ONGC.NS', 'OFSS.NS', 'TORNTPOWER.NS', 'ALKEM.NS', 'APLLTD.NS', 'WHIRLPOOL.NS', 'NMDC.NS', 'JPPOWER.NS', 'IFCI.NS', 'TANLA.NS', 'SKFINDIA.NS', 'BIOCON.NS', 'SPARC.NS', 'BLUEDART.NS', 'SRF.NS', 'DIXON.NS', 'SHRIRAMFIN.NS', 'AJANTPHARM.NS', 'ADANIPOWER.NS', 'HEG.NS', 'POLYMED.NS', 'JUBLPHARMA.NS', 'BALAMINES.NS', 'TORNTPHARM.NS']
Retained stocks: []
Expelled stocks: []
New additions: ['TORNTPOWER.NS', 'ADANIPOWER.NS', 'SKFINDIA.NS', 'BALAMINES.NS', 'JUSTDIAL.NS', 'SHRIRAMFIN.NS', 'AJANTPHARM.NS', 'BLUEDART.NS', 'WHIRLPOOL.NS', 'DIXON.NS', 'JUBLPHARMA.NS', 'OFSS.NS', '

In [91]:
from typing import Tuple

class Strategy():
    def __init__(self):
        # 1. Read CSV with Date Parsing
        self.signalsData = pd.read_csv(
            'auxilary/backtester_weights.csv', 
            index_col=0,           # Set Date as index
            parse_dates=True,      # Convert index to datetime objects
            na_values=['nan', 'NaN', ''],
            keep_default_na=True
        )
        
        # 2. Normalize Index (Remove Time) to match the Backtester's data
        # This ensures '2020-02-06 09:15:00' matches '2020-02-06 00:00:00'
        self.signalsData.index = pd.to_datetime(self.signalsData.index).normalize()
        self.signalsData = self.signalsData.shift(1)

    def get_signals(self, tradingState: dict) -> Tuple[list, str]:
        # 3. Get the current date from the Backtester
        current_ts = tradingState['current_timestamp']
        
        # 4. Use .loc (Date Lookup) instead of .iloc (Row Lookup)
        try:
            signal = self.signalsData.loc[current_ts]
            
            # If duplicates exist for a date, take the last one (optional safety)
            if isinstance(signal, pd.DataFrame):
                signal = signal.iloc[-1]
                
        except KeyError:
            # If the date is missing in weights, return NaNs (No Action)
            signal = pd.Series(float('nan'), index=self.signalsData.columns)

        # Ensure signal is a Series (handle edge cases)
        if not isinstance(signal, pd.Series):
             signal = pd.Series(signal)

        # We don't need traderData (index counter) anymore
        traderData = tradingState['traderData'] 
        
        return signal, traderData

In [92]:
class Backtester:
    def __init__(self, data: pd.DataFrame, initial_value: float, start_date):
        self.data = data
        self.initialvalue = initial_value
        self.portfolio_value = initial_value
        self.cash = initial_value
        self.investment = 0.0
        self.current_index = 1
        tickers = data.columns.get_level_values(0).unique()
        self.positions = pd.Series(0, index=tickers)
        self.all_positions = pd.DataFrame(columns=tickers)
        self.tradingState = {}
        self.all_signals = pd.DataFrame(columns=tickers)
        self.startdate = start_date

    def calculate_positions(self, signal: pd.Series, value, open=True) -> pd.Series:
        # --- Checks (Unchanged) ---
        if (signal < 0).any():
            raise ValueError(f'For timestamp {self.data.index[self.current_index]}, signal contains negative values: {signal[signal < 0]}')
        if not isinstance(signal, pd.Series):
            raise TypeError(f'For timestamp {self.data.index[self.current_index]}, signal must be a pandas Series, got {type(signal)}')
        if abs(signal).sum() - 1 > 1e-6:
            raise ValueError(f'For timestamp {self.data.index[self.current_index]} the sum of the abs(signals) must not be greater than 1, got {abs(signal).sum()}')

        prices = (
            self.data.xs('open', level=1, axis=1).iloc[self.current_index]
            if open
            else self.data.xs('close', level=1, axis=1).iloc[self.current_index]
        )
        prices = prices.reindex(signal.index)

        aligned_positions = self.positions.reindex(signal.index).fillna(0)

        nan_index = signal.isna()

        value -= (aligned_positions[nan_index] * prices[nan_index]).sum()

        float_shares = (signal.replace(0, np.nan) * value) / prices.replace(0, np.nan)
        float_shares = float_shares.replace([np.inf, -np.inf], 0).fillna(0)

        new_positions = pd.Series(0, index=float_shares.index, dtype=int)
        longs = float_shares > 0
        shorts = float_shares < 0

        new_positions[longs] = np.floor(float_shares[longs]).astype(int)
        new_positions[shorts] = np.ceil(float_shares[shorts]).astype(int)

        new_positions[nan_index] = aligned_positions[nan_index]

        return new_positions

    def calculate_cash(self, positions: pd.Series, open=True) -> float:
        index = self.current_index
        price = self.data.xs('open', level=1, axis=1).iloc[index] if open else self.data.xs('close', level=1, axis=1).iloc[index]
        return self.portfolio_value - (abs(positions) * price).sum()

    def update_investment(self, positions: pd.Series, new_day=False) -> float:
        index = self.current_index
        price1 = self.data.xs('close', level=1, axis=1).iloc[index - 1] if new_day else self.data.xs('open', level=1, axis=1).iloc[index]
        price2 = self.data.xs('open', level=1, axis=1).iloc[index] if new_day else self.data.xs('close', level=1, axis=1).iloc[index]
        return (positions * (price2 - price1)).sum() + self.investment

    def run(self, mode='normal'):
        output_signals_path = 'results/signals.csv'
        output_positions_path = 'results/positions.csv'

        import os
        if not os.path.exists('results'):
            os.makedirs('results')
            print("Created 'results' directory for output files.")
        
        # Validate weights file exists
        weights_file = '.cache/backtester_weights_cache.csv' if mode == 'looptest' else 'auxilary/backtester_weights.csv'
        if not os.path.exists(weights_file):
            raise FileNotFoundError(f"Weights file not found: {weights_file}. Make sure weights_generation() was called first.")

        print(f"Starting backtest (NumPy engine, mode='{mode}')...")
        if mode == 'looptest':
            strategy = Strategy_looptest()
        else:
            strategy = Strategy()

        tickers = self.data.columns.get_level_values(0).unique()
        n_assets = len(tickers)
        n_steps = len(self.data.index)

        open_df = self.data.xs('open', level=1, axis=1).loc[:, tickers]
        close_df = self.data.xs('close', level=1, axis=1).loc[:, tickers]
        open_px = open_df.to_numpy(dtype=float)
        close_px = close_df.to_numpy(dtype=float)

        pos_array = np.zeros(n_assets, dtype=np.int64)
        cash = float(self.initialvalue)

        all_pos_history = np.zeros((n_steps, n_assets), dtype=np.int64)
        all_sig_history = np.full((n_steps, n_assets), np.nan, dtype=float)

        traderData = 0
        for i in tqdm.tqdm(range(0, n_steps)):
            ts = self.data.index[i]

            # Keep Strategy() contract: pull one weight row per step from the weights CSV
            self.tradingState = {
                'investment': self.investment,
                'cash': self.cash,
                'current_timestamp': ts,
                'traderData': traderData,
                'positions': self.positions,
            }
            signal, traderData = strategy.get_signals(self.tradingState)
            if signal is None:
                raise ValueError(f'For timestamp {ts}, signal is None')
            signal = signal.reindex(tickers)
            weights_row = signal.to_numpy(dtype=float)
            all_sig_history[i] = weights_row

            # --- Execution at Open[i], valuation at Close[i] ---
            curr_open = open_px[i]
            curr_close = close_px[i]

            # Mark-to-market at open before trading
            portfolio_value_open = cash + float(np.sum(pos_array * curr_open))

            update_mask = ~np.isnan(weights_row)
            if np.any(update_mask):
                # Validate long-only weights constraint (same idea as calculate_positions)
                if np.nansum(np.abs(weights_row)) - 1 > 1e-6:
                    raise ValueError(
                        f'For timestamp {ts} the sum of abs(weights) must not be greater than 1, got {np.nansum(np.abs(weights_row))}'
                    )
                if np.nanmin(weights_row) < -1e-12:
                    raise ValueError(f'For timestamp {ts}, weights contain negative values.')

                # Reserve value for holdings that are not being updated today (NaN => keep)
                reserved_value = float(np.sum(pos_array[~update_mask] * curr_open[~update_mask]))
                available_value = portfolio_value_open - reserved_value
                if available_value < 0:
                    available_value = 0.0

                target_weights = np.nan_to_num(weights_row[update_mask], nan=0.0)
                target_prices = curr_open[update_mask]

                # Allocate only the available value across updated tickers
                dollars = target_weights * available_value

                with np.errstate(divide='ignore', invalid='ignore'):
                    shares = np.floor(np.divide(dollars, target_prices, out=np.zeros_like(dollars), where=target_prices != 0.0))
                    shares = np.nan_to_num(shares, nan=0.0, posinf=0.0, neginf=0.0).astype(np.int64)

                pos_array[update_mask] = shares

            # Cash after trades at open
            cash = portfolio_value_open - float(np.sum(pos_array * curr_open))

            # End-of-day valuation at close
            investment_close = float(np.sum(pos_array * curr_close))
            portfolio_value_close = cash + investment_close

            all_pos_history[i] = pos_array
            self.current_index = i
            self.cash = cash
            self.investment = investment_close
            self.portfolio_value = portfolio_value_close

        # Store outputs as DataFrames (same shape as before)
        self.positions = pd.Series(pos_array, index=tickers)
        self.all_positions = pd.DataFrame(all_pos_history, index=self.data.index, columns=tickers)
        self.all_signals = pd.DataFrame(all_sig_history, index=self.data.index, columns=tickers)

        try:
            self.all_signals.to_csv(output_signals_path)
            self.all_positions.to_csv(output_positions_path)
            tqdm.tqdm.write(f"\n--- Checkpoint Saved progress to CSV files. ---")
        except Exception as e:
            tqdm.tqdm.write(f"\n--- Could not save checkpoint. Reason: {e} ---")

    def vectorbt_run(self, rolling_period=None, mean_period=None, rolling_threshold=None, dilute_to=None, mode='normal'):
        # These parameters are provided for optimization loops but not actively used in current implementation
        # They can be extended in future versions for parameter-specific logic
        if self.startdate is None or pd.isna(self.startdate):
            filtered_positions = self.all_positions
        else:
            filtered_positions = self.all_positions[self.all_positions.index >= self.startdate]
            if filtered_positions.empty:
                filtered_positions = self.all_positions

        open_prices = self.data.xs('open', level=1, axis=1).loc[filtered_positions.index, filtered_positions.columns]
        close_prices = self.data.xs('close', level=1, axis=1).loc[filtered_positions.index, filtered_positions.columns]

        positions_to_use = filtered_positions
        order_size = positions_to_use.diff()
        order_size.iloc[0] = positions_to_use.iloc[0]
        order_size = order_size.astype(int)

        order_size = order_size.reindex(index=open_prices.index, columns=open_prices.columns).fillna(0)
        order_size = order_size.mask(order_size == 0)

        portfolio = vbt.Portfolio.from_orders(
            close=close_prices,
            size=order_size,
            price=open_prices,
            init_cash=self.initialvalue,
            freq='1D',
            cash_sharing=True,
            call_seq='auto',
            log=True,
        )
        print(f"Initial Amount= {portfolio.init_cash}")

        benchmark_df = pd.read_csv('src/benchmark.csv', index_col=0, parse_dates=True)
        benchmark_df = benchmark_df[START_DATE:END_DATE]

        start_price = benchmark_df['close'].iloc[0]
        end_price = benchmark_df['close'].iloc[-1]
        benchmark_return = ((end_price - start_price) / start_price) * 100

        stats_eq = portfolio.stats()
        stats_df = stats_eq.to_frame(name='Value').reset_index()
        stats_df.columns = ['Metric', 'Value']

        stats_df = stats_df[stats_df['Metric'] != 'Benchmark Return [%]'].reset_index(drop=True)
        benchmark_row = pd.DataFrame({
            'Metric': ['Benchmark Return [%]'],
            'Value': [benchmark_return]
        })

        stats_df_top = stats_df.iloc[:6]
        stats_df_bottom = stats_df.iloc[6:]
        stats_df = pd.concat([stats_df_top, benchmark_row, stats_df_bottom], ignore_index=True)

        # Define custom date ranges for period analysis
        custom_periods = [
            (None, '2020-03-23'),
            ('2020-03-23', '2021-10-18'),
            ('2021-10-18', '2023-03-27'),
            ('2023-03-27', '2024-09-24'),
            ('2024-09-24', '2025-09-30')
        ]

        period_metrics = []
        pf_value = portfolio.value()

        for start_str, end_str in custom_periods:
            try:
                if start_str is None:
                    s_date = pf_value.index[0]
                    start_label = "Inception"
                else:
                    s_date = pd.Timestamp(start_str)
                    start_label = start_str
                
                e_date = pd.Timestamp(end_str)
                sub_period = pf_value[s_date:e_date]

                if not sub_period.empty:
                    val_start = sub_period.iloc[0]
                    val_end = sub_period.iloc[-1]
                    period_ret = ((val_end - val_start) / val_start) * 100
                    
                    period_metrics.append({
                        'Metric': f'Return: {start_label} -> {end_str} [%]',
                        'Value': period_ret
                    })
                else:
                    period_metrics.append({
                        'Metric': f'Return: {start_label} -> {end_str} [%]',
                        'Value': 0.0
                    })

            except Exception as e:
                print(f"Skipping period {start_str} to {end_str}: {e}")

        if period_metrics:
            period_df = pd.DataFrame(period_metrics)
            stats_df = pd.concat([stats_df, period_df], ignore_index=True)

        portfolio.assets().to_csv('results/assets.csv')
        portfolio.orders.records_readable.to_csv('results/log.csv')

        df = pd.concat([portfolio.value(), portfolio.asset_value(), portfolio.cash()], axis=1)
        df.columns = ['portfolio', 'investment', 'cash']
        df.to_csv('results/portfolio.csv')

        return portfolio, stats_df

In [93]:
data = pd.read_csv(
    'src/all_ohlcv_clipped.csv',
    index_col=0, parse_dates=True, header=[0,1]
)
backtester = Backtester(data, INITIAL_CAPITAL,TRADING_START_DATE)
backtester.run()
print(f"Final Portfolio value= {backtester.portfolio_value}")
pf,stats_df_csv = backtester.vectorbt_run()
print(stats_df_csv)


Starting backtest (NumPy engine, mode='normal')...


100%|██████████| 2231/2231 [00:00<00:00, 19041.66it/s]



--- Checkpoint Saved progress to CSV files. ---
Final Portfolio value= 100000.0
Initial Amount= 100000.0
                                  Metric                Value
0                                  Start  2018-01-01 09:15:00
1                                    End  2025-12-31 09:15:00
2                                 Period   1983 days 00:00:00
3                            Start Value             100000.0
4                              End Value             100000.0
5                       Total Return [%]                  0.0
6                   Benchmark Return [%]           209.540164
7                 Max Gross Exposure [%]                  0.0
8                        Total Fees Paid                  0.0
9                       Max Drawdown [%]                  NaN
10                 Max Drawdown Duration                  NaT
11                          Total Trades                    0
12                   Total Closed Trades                    0
13                     Tot

In [94]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

returns = eq_curve.pct_change().fillna(0)
returns=returns[TRADING_START_DATE:END_DATE]
cum_max = eq_curve.cummax()
drawdown = (eq_curve - cum_max) / cum_max
mean_ret = returns.mean()
median_ret = returns.median()

fig = make_subplots(rows=1, cols=2, subplot_titles=('Daily Returns', 'Drawdown Curve'))
fig.add_trace(
    go.Histogram(
        x=returns,
        nbinsx=50,
        marker_color='orange',
        marker_line_color='white',
        marker_line_width=1,
        opacity=0.8,
        name='Returns'
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=drawdown.index,
        y=drawdown.values,
        mode='lines',
        line=dict(color='red', width=2),
        name='Drawdown'
    ),
    row=1, col=2
)
fig.update_xaxes(
    title_text='Return',
    row=1, col=1,
    nticks=20,
    showgrid=True
)
fig.update_yaxes(
    title_text='Frequency',
    row=1, col=1,
    showgrid=True
)
fig.update_xaxes(
    title_text='Date',
    row=1, col=2,
    showgrid=False
)
fig.update_yaxes(
    title_text='Drawdown',
    row=1, col=2,
    showgrid=True
)
fig.update_layout(
    title_text='Returns Distribution & Drawdown',
    bargap=0.1,               
    template='plotly_white',
    showlegend=False,
    width=900,
    height=400
)

fig.show()
